# 🎬 Movie Recommender System
### Content-Based Filtering using Cosine Similarity

**How it works:**
1. Load and clean the movie dataset
2. Create tags by combining genres, keywords, cast, crew and overview
3. Convert tags into vectors using CountVectorizer
4. Calculate Cosine Similarity between all movies
5. Save the model using pickle for use in the Streamlit app

In [1]:
# -----------------------------------------------
# STEP 1: Import all required libraries
# -----------------------------------------------
import numpy as np
import pandas as pd
import ast       # to convert string representation of list into actual list
import pickle    # to save the model
import os        # to create folders

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# -----------------------------------------------
# STEP 2: Load the dataset CSV files
# Make sure both CSV files are in your project folder
# -----------------------------------------------
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print("Movies  shape :", movies.shape)
print("Credits shape :", credits.shape)

Movies  shape : (4803, 20)
Credits shape : (4803, 4)


In [3]:
# -----------------------------------------------
# STEP 3: Merge both tables on the 'title' column
# -----------------------------------------------
movies = movies.merge(credits, on='title')

print("Merged shape:", movies.shape)
movies.head(2)

Merged shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [4]:
# -----------------------------------------------
# STEP 4: Keep only the useful columns
# -----------------------------------------------
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

movies.dropna(inplace=True)   # remove rows with missing values

print("Shape after selecting columns:", movies.shape)
movies.head(2)

Shape after selecting columns: (4806, 7)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [5]:
# -----------------------------------------------
# STEP 5: Create helper functions to extract text from JSON-like strings
# -----------------------------------------------

# For genres and keywords — extract all names
def convert(text):
    result = []
    for item in ast.literal_eval(text):
        result.append(item['name'])
    return result

# For cast — extract only top 3 actor names
def convert_cast(text):
    result = []
    for i, item in enumerate(ast.literal_eval(text)):
        if i == 3:      # stop after 3 actors
            break
        result.append(item['name'])
    return result

# For crew — extract only the Director's name
def fetch_director(text):
    result = []
    for item in ast.literal_eval(text):
        if item['job'] == 'Director':
            result.append(item['name'])
    return result

print("Helper functions created successfully")

Helper functions created successfully


In [6]:
# -----------------------------------------------
# STEP 6: Apply helper functions to each column
# -----------------------------------------------
movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print("All columns converted successfully")
movies.head(2)

All columns converted successfully


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]


In [7]:
# -----------------------------------------------
# STEP 7: Remove spaces inside multi-word names
# Example: 'Sam Worthington' → 'SamWorthington'
# This prevents 'Sam' and 'Worthington' from being counted separately
# -----------------------------------------------
def collapse(word_list):
    return [word.replace(" ", "") for word in word_list]

movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)

print("Spaces removed from multi-word names")
movies.head(2)

Spaces removed from multi-word names


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


In [8]:
# -----------------------------------------------
# STEP 8: Combine all columns into one 'tags' column
# Tags = overview + genres + keywords + cast + crew
# -----------------------------------------------
movies['tags'] = (movies['overview']
                + movies['genres']
                + movies['keywords']
                + movies['cast']
                + movies['crew'])

# Keep only the required columns
new_df = movies[['movie_id', 'title', 'tags']].copy()

# Join the list into a single string and convert to lowercase
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

print("Tags column created successfully")
new_df.head(3)

Tags column created successfully


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


In [9]:
# -----------------------------------------------
# STEP 9: Convert text tags into numerical vectors
# CountVectorizer counts the frequency of each word
# max_features = 5000  → use only top 5000 words
# stop_words = english → ignore common words like 'the', 'is', 'a'
# -----------------------------------------------
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

print("Vector shape:", vectors.shape)
print("Meaning:", vectors.shape[0], "movies and", vectors.shape[1], "unique words")

Vector shape: (4806, 5000)
Meaning: 4806 movies and 5000 unique words


In [10]:
# -----------------------------------------------
# STEP 10: Calculate Cosine Similarity between all movies
# Score ranges from 0 to 1
# 1 = very similar,  0 = completely different
# -----------------------------------------------
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)

Similarity matrix shape: (4806, 4806)


In [11]:
# -----------------------------------------------
# STEP 11: Test the recommendation function
# -----------------------------------------------
def recommend(movie):
    # Get the index of the selected movie
    index = new_df[new_df['title'] == movie].index[0]

    # Sort movies by similarity score (highest first)
    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])

    print(f"Movies similar to '{movie}':")
    print("-" * 35)
    for i in distances[1:6]:    # Top 5 results (skip index 0 — that's the movie itself)
        print("👉", new_df.iloc[i[0]].title)

# Test with a sample movie
recommend('Batman Begins')

Movies similar to 'Batman Begins':
-----------------------------------
👉 The Dark Knight
👉 The Dark Knight Rises
👉 Batman
👉 Batman
👉 Batman & Robin


In [12]:
# -----------------------------------------------
# STEP 12: Save the model files using pickle
# app.py will load these files to show recommendations
# -----------------------------------------------
os.makedirs('model', exist_ok=True)   # create 'model' folder (skip if already exists)

pickle.dump(new_df,     open('model/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('model/similarity.pkl', 'wb'))

print("model/movie_list.pkl saved successfully")
print("model/similarity.pkl saved successfully")
print()
print("Now run: streamlit run app.py")

model/movie_list.pkl saved successfully
model/similarity.pkl saved successfully

Now run: streamlit run app.py
